In [1]:
from pathlib import Path
import re
import numpy as np

# User settings
workdir = Path(".")          # local
target_T = 400.0             # the temperature you want
output_name = "POSCAR_near_400K"

OSZICAR = workdir / "OSZICAR"
XDATCAR = workdir / "XDATCAR"

# Read temperatures from OSZICAR
def read_temperatures_from_oszicar(oszicar_path):
    steps = []
    temps = []

    temp_pattern = re.compile(
        r"T=\s*([+-]?\d+(?:\.\d*)?(?:[Ee][+-]?\d+)?)"
    )

    with open(oszicar_path, "r", errors="ignore") as f:
        for line in f:
            m = temp_pattern.search(line)
            if m is None:
                continue

            step_match = re.match(r"\s*(\d+)", line)
            step = int(step_match.group(1)) if step_match else len(temps) + 1

            steps.append(step)
            temps.append(float(m.group(1)))

    if len(temps) == 0:
        raise ValueError("No temperature information `T=` was found in OSZICAR.")

    return np.array(steps), np.array(temps)

# Read XDATCAR frames
def is_counts_line(line):
    tokens = line.split()
    if len(tokens) == 0:
        return False
    return all(tok.isdigit() for tok in tokens)


def read_xdatcar(xdatcar_path):
    with open(xdatcar_path, "r", errors="ignore") as f:
        lines = f.readlines()

    title = lines[0].rstrip()
    scale = lines[1].rstrip()
    lattice = [lines[i].rstrip() for i in range(2, 5)]

    # VASP 5/6 usually has element symbols line, VASP 4 may not.
    if is_counts_line(lines[5]):
        symbols = None
        counts_line_index = 5
    else:
        symbols = lines[5].split()
        counts_line_index = 6

    counts = [int(x) for x in lines[counts_line_index].split()]
    natoms = sum(counts)

    frames = []
    config_numbers = []
    coord_modes = []

    i = counts_line_index + 1
    while i < len(lines):
        line = lines[i].strip()

        if "configuration" in line.lower():
            # Example:
            # Direct configuration=     123
            num_match = re.search(r"configuration\s*=\s*(\d+)", line, re.I)
            config_num = int(num_match.group(1)) if num_match else len(frames) + 1

            if line.lower().startswith("cart"):
                coord_mode = "Cartesian"
            else:
                coord_mode = "Direct"

            coords = [lines[i + 1 + j].rstrip() for j in range(natoms)]

            frames.append(coords)
            config_numbers.append(config_num)
            coord_modes.append(coord_mode)

            i += natoms + 1
        else:
            i += 1

    if len(frames) == 0:
        raise ValueError("No ionic configurations were found in XDATCAR.")

    header = {
        "title": title,
        "scale": scale,
        "lattice": lattice,
        "symbols": symbols,
        "counts": counts,
    }

    return header, frames, config_numbers, coord_modes

# Write POSCAR
def write_poscar(header, coords, coord_mode, output_path):
    lines = []

    lines.append(header["title"] + "\n")
    lines.append(header["scale"] + "\n")

    for vec in header["lattice"]:
        lines.append(vec + "\n")

    if header["symbols"] is not None:
        lines.append(" ".join(header["symbols"]) + "\n")

    lines.append(" ".join(str(x) for x in header["counts"]) + "\n")
    lines.append(coord_mode + "\n")

    for c in coords:
        lines.append(c + "\n")

    with open(output_path, "w") as f:
        f.writelines(lines)

# Main
steps, temps = read_temperatures_from_oszicar(OSZICAR)
header, frames, config_numbers, coord_modes = read_xdatcar(XDATCAR)

n_temps = len(temps)
n_frames = len(frames)

print(f"Number of MD temperatures in OSZICAR: {n_temps}")
print(f"Number of structures in XDATCAR:      {n_frames}")

# Most common case: OSZICAR temperature count == XDATCAR frame count
if n_temps == n_frames:
    frame_temp = temps
    frame_step = steps

# If XDATCAR was written less frequently, for example NBLOCK > 1
elif n_temps > n_frames and n_temps % n_frames == 0:
    stride = n_temps // n_frames
    print(f"Detected possible XDATCAR output stride: every {stride} MD steps")

    oszicar_indices = np.arange(stride - 1, n_temps, stride)
    frame_temp = temps[oszicar_indices]
    frame_step = steps[oszicar_indices]

else:
    raise ValueError(
        "The number of OSZICAR temperatures and XDATCAR frames do not match. "
        "This may be due to NBLOCK or an incomplete run. "
        "Please check XDATCAR output frequency."
    )

best_frame_index = int(np.argmin(np.abs(frame_temp - target_T)))

best_T = frame_temp[best_frame_index]
best_step = frame_step[best_frame_index]
best_config = config_numbers[best_frame_index]
best_diff = abs(best_T - target_T)

output_path = workdir / output_name

write_poscar(
    header=header,
    coords=frames[best_frame_index],
    coord_mode=coord_modes[best_frame_index],
    output_path=output_path,
)

print()
print("Done.")
print(f"Target temperature:      {target_T:.2f} K")
print(f"Closest temperature:     {best_T:.2f} K")
print(f"Temperature difference:  {best_diff:.2f} K")
print(f"MD step from OSZICAR:    {best_step}")
print(f"XDATCAR configuration:   {best_config}")
print(f"Output POSCAR:           {output_path}")

Number of MD temperatures in OSZICAR: 5500
Number of structures in XDATCAR:      550
Detected possible XDATCAR output stride: every 10 MD steps

Done.
Target temperature:      400.00 K
Closest temperature:     400.00 K
Temperature difference:  0.00 K
MD step from OSZICAR:    220
XDATCAR configuration:   220
Output POSCAR:           POSCAR_near_400K
